# SnowHealth Demand Forecasting - Feature Engineering
This notebook transforms raw transaction data into ML-ready features and registers them in the Snowflake Feature Store as a **managed Dynamic Table** with **10-minute refresh** and **online serving** for low-latency lookups.

In [ ]:
# Initialize session and imports
import sys, os
work_dir = os.getcwd()
sys.path.insert(0, os.path.join(work_dir, 'source'))

from snowflake.snowpark.context import get_active_session
import config as cfg
from feature_engineering import (
    engineer_features, get_feature_columns,
    add_calendar_features, add_cyclical_features,
    add_lag_features, add_interaction_features
)
import pandas as pd
import numpy as np

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_RAW_SCHEMA}").collect()
print(f"Session ready: {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_RAW_SCHEMA}")

In [ ]:
# Load raw data
df_raw = session.table(cfg.SNOWHEALTH_TRANSACTIONS_TABLE).to_pandas()
print(f"Loaded {len(df_raw):,} rows, {len(df_raw.columns)} columns")
print(f"Target stats: mean={df_raw['UNITS_SOLD'].mean():.1f}, median={df_raw['UNITS_SOLD'].median():.0f}, std={df_raw['UNITS_SOLD'].std():.1f}")

In [ ]:
# Step 1: Calendar features
df_feat = add_calendar_features(df_raw)
print("Calendar features added:", ['MONTH', 'QUARTER', 'WEEK_OF_YEAR', 'IS_WEEKEND_INT', 'IS_HOLIDAY_INT', 'IS_PROMOTION_INT'])

# Step 2: Cyclical encoding
df_feat = add_cyclical_features(df_feat)
print("Cyclical features added:", ['MONTH_SIN', 'MONTH_COS', 'DOW_SIN', 'DOW_COS'])

# Step 3: Lag and rolling features
df_feat = add_lag_features(df_feat)
print("Lag features added:", ['ROLLING_7D_AVG_UNITS', 'ROLLING_30D_AVG_UNITS', 'LAG_7D_UNITS', 'LAG_14D_UNITS', 'LAG_30D_UNITS'])

# Step 4: Interaction features
df_feat = add_interaction_features(df_feat)
print("Interaction features added:", ['PRICE_TIMES_PROMO', 'INVENTORY_TURNOVER_RATIO', 'SEASONAL_DEMAND_INDEX'])

print(f"\nTotal features: {len(df_feat.columns)} columns")

In [ ]:
# Visualize feature distributions
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
feats_to_plot = ['ROLLING_7D_AVG_UNITS', 'ROLLING_30D_AVG_UNITS', 'SEASONAL_DEMAND_INDEX',
                 'MONTH_SIN', 'INVENTORY_TURNOVER_RATIO', 'PRICE_TIMES_PROMO']
for i, feat in enumerate(feats_to_plot):
    ax = axes[i//3, i%3]
    ax.hist(df_feat[feat].dropna(), bins=40, color='#cc0000', edgecolor='white', alpha=0.8)
    ax.set_title(feat, fontsize=11)
plt.suptitle('Engineered Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Save engineered features to Snowflake
keep_cols = ['TRANSACTION_ID', 'TRANSACTION_DATE', 'STORE_ID', 'REGION', 'SKU_ID',
             'PRODUCT_NAME', 'PRODUCT_CATEGORY', 'FULFILLMENT_CHANNEL',
             'UNITS_SOLD', 'UNIT_PRICE', 'PROMOTION_DISCOUNT_PCT',
             'DAY_OF_WEEK', 'IS_WEEKEND_INT', 'IS_HOLIDAY_INT', 'IS_PROMOTION_INT',
             'TEMPERATURE_F', 'FLU_INDEX', 'STORE_SIZE_SQFT', 'STAFF_COUNT',
             'COMPETITOR_PROXIMITY_MILES', 'INVENTORY_ON_HAND',
             'DAYS_SUPPLY_ON_HAND', 'LEAD_TIME_DAYS',
             'MONTH', 'QUARTER', 'WEEK_OF_YEAR',
             'MONTH_SIN', 'MONTH_COS', 'DOW_SIN', 'DOW_COS',
             'ROLLING_7D_AVG_UNITS', 'ROLLING_30D_AVG_UNITS',
             'LAG_7D_UNITS', 'LAG_14D_UNITS', 'LAG_30D_UNITS',
             'PRICE_TIMES_PROMO', 'INVENTORY_TURNOVER_RATIO',
             'SEASONAL_DEMAND_INDEX']

df_save = df_feat[keep_cols].copy()

# Create EVENT_TIMESTAMP (TIMESTAMP_NTZ) from TRANSACTION_DATE for online serving
df_save['TRANSACTION_DATE'] = pd.to_datetime(df_save['TRANSACTION_DATE'])
df_save['EVENT_TIMESTAMP'] = df_save['TRANSACTION_DATE']

# Convert types for Snowflake compatibility
for col in df_save.select_dtypes(include=['int8', 'int16', 'int32']).columns:
    df_save[col] = df_save[col].astype('int64')
for col in df_save.select_dtypes(include=['float32']).columns:
    df_save[col] = df_save[col].astype('float64')

snow_df = session.create_dataframe(df_save)
snow_df.write.mode('overwrite').save_as_table(f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES")

# Enable change tracking (required for managed/Dynamic Table feature views)
session.sql(f"""
    ALTER TABLE {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES
    SET CHANGE_TRACKING = TRUE
""").collect()

print(f"Saved {len(df_save):,} rows to {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES")
print(f"Columns: {len(df_save.columns)} (includes EVENT_TIMESTAMP for online serving)")
print(f"Change tracking: enabled")

In [ ]:
# Initialize Feature Store and register entity
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode, OnlineConfig

fs = FeatureStore(
    session=session,
    database=cfg.SNOWHEALTH_DATABASE,
    name=cfg.SNOWHEALTH_FEATURES_SCHEMA,
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print("Feature Store initialized")

# Register entity (idempotent - safe to re-run)
try:
    snowhealth_entity = fs.get_entity(cfg.SNOWHEALTH_ENTITY_NAME)
    print(f"Entity already exists: {cfg.SNOWHEALTH_ENTITY_NAME}")
except:
    snowhealth_entity = Entity(
        name=cfg.SNOWHEALTH_ENTITY_NAME,
        join_keys=["SKU_ID", "STORE_ID"],
        desc="SNOWHEALTH product-store combination for demand forecasting"
    )
    fs.register_entity(snowhealth_entity)
    print(f"Entity registered: {cfg.SNOWHEALTH_ENTITY_NAME}")

print(fs.list_entities().to_pandas()[['NAME', 'JOIN_KEYS']])

In [ ]:
# Register managed Feature View with 10-minute refresh
# Delete existing FV if present (safe for re-runs)
try:
    fs.delete_feature_view(cfg.SNOWHEALTH_FEATURE_VIEW_NAME, cfg.SNOWHEALTH_FEATURE_VIEW_VERSION)
    print(f"Deleted existing FV: {cfg.SNOWHEALTH_FEATURE_VIEW_NAME} v{cfg.SNOWHEALTH_FEATURE_VIEW_VERSION}")
except:
    pass

feature_df = session.table(f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES")

fv = FeatureView(
    name=cfg.SNOWHEALTH_FEATURE_VIEW_NAME,
    entities=[snowhealth_entity],
    feature_df=feature_df,
    timestamp_col="EVENT_TIMESTAMP",
    refresh_freq="10 minutes",
    desc="SNOWHEALTH demand features - managed DT with 10-min refresh and online serving",
)

fv = fv.attach_feature_desc({
    "UNITS_SOLD": "Target: number of units sold",
    "ROLLING_7D_AVG_UNITS": "7-day rolling average of units sold per SKU-store",
    "ROLLING_30D_AVG_UNITS": "30-day rolling average of units sold per SKU-store",
    "SEASONAL_DEMAND_INDEX": "Monthly seasonal demand multiplier",
    "INVENTORY_TURNOVER_RATIO": "Ratio of units sold to inventory on hand",
    "PRICE_TIMES_PROMO": "Interaction: unit price * promotion discount",
    "MONTH_SIN": "Cyclical encoding of month (sine component)",
    "MONTH_COS": "Cyclical encoding of month (cosine component)",
})

print("Registering managed feature view (creates a Dynamic Table)...")
registered_fv = fs.register_feature_view(
    feature_view=fv,
    version=cfg.SNOWHEALTH_FEATURE_VIEW_VERSION,
    block=True,
)
print(f"Feature View registered: {cfg.SNOWHEALTH_FEATURE_VIEW_NAME} v{cfg.SNOWHEALTH_FEATURE_VIEW_VERSION}")
print(f"Status: {registered_fv.status}")
print(f"Type: Managed (Dynamic Table) | Refresh: every 10 minutes")

In [ ]:
# Enable online feature serving for low-latency lookups
from snowflake.ml.feature_store.feature_view import StoreType

online_config = OnlineConfig(enable=True, target_lag="1m")
updated_fv = fs.update_feature_view(
    name=cfg.SNOWHEALTH_FEATURE_VIEW_NAME,
    version=cfg.SNOWHEALTH_FEATURE_VIEW_VERSION,
    online_config=online_config,
)
print(f"Online serving enabled: {updated_fv.online}")
print(f"Online target lag: 1 minute")

# Verify online lookup
import time
print("\nWaiting for online feature table to provision...")
for attempt in range(12):
    try:
        result = fs.read_feature_view(
            cfg.SNOWHEALTH_FEATURE_VIEW_NAME, cfg.SNOWHEALTH_FEATURE_VIEW_VERSION,
            keys=[["RX-001", "SH-1001"]],
            store_type=StoreType.ONLINE,
        )
        online_data = result.to_pandas()
        print(f"Online store ready! Retrieved {len(online_data)} row(s) for key (RX-001, SH-1001)")
        break
    except Exception as e:
        if attempt < 11:
            print(f"  Provisioning... ({attempt+1}/12)")
            time.sleep(10)
        else:
            print(f"Still provisioning - will be ready shortly: {e}")

In [ ]:
# Summary
final_fv = fs.get_feature_view(cfg.SNOWHEALTH_FEATURE_VIEW_NAME, cfg.SNOWHEALTH_FEATURE_VIEW_VERSION)
print("=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)
print(f"Features table: {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES")
print(f"Feature Store:  {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}")
print(f"Entity:         {cfg.SNOWHEALTH_ENTITY_NAME} (join keys: SKU_ID, STORE_ID)")
print(f"Feature View:   {cfg.SNOWHEALTH_FEATURE_VIEW_NAME} v{cfg.SNOWHEALTH_FEATURE_VIEW_VERSION}")
print(f"Type:           Managed (Dynamic Table)")
print(f"Refresh:        Every 10 minutes")
print(f"Online:         {final_fv.online}")
print(f"Features:       {len(final_fv.feature_names)}")
print("=" * 60)
print("Proceed to 03_model_training.ipynb")